In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA
from tqdm.notebook import tqdm
from datetime import datetime
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# --- 1. Database & Path Setup ---
db_connection_str = "mysql+pymysql://root:@127.0.0.1/trading_system"
db_engine = create_engine(db_connection_str)

base_dir = Path.cwd().parent
output_dir = base_dir / "4_Results" / "Full_Ablation_Runs"
output_dir.mkdir(parents=True, exist_ok=True)

# --- 2. Define the ISOLATED Feature Sets ---
base_features = [
    'prev_close', 'ma_3', 'ma_5', 'ma_10', 'volatility', 
    'rsi', 'momentum_3', 'momentum_5', 'volume'
]

feature_variants = {
    "VADER":      base_features + ['vader_compound'],
    "TextBlob":   base_features + ['textblob_polarity'],
    "FinBERT":    base_features + ['finbert_compound'],
}

# --- 3. Model Definitions ---
# Helper to get fresh models for every iteration
def get_models():
    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=42),
        "Lasso": Lasso(random_state=42),
        # Using n_jobs=1 to prevent thread contention with the main loop
        "RandomForest": RandomForestRegressor(n_estimators=50, max_depth=10, n_jobs=1, random_state=42),
        "XGBoost": xgb.XGBRegressor(n_estimators=50, max_depth=3, learning_rate=0.1, n_jobs=1, random_state=42),
        "SVR": SVR(kernel='rbf', C=1.0),
        "MLP": MLPRegressor(hidden_layer_sizes=(50,), max_iter=200, early_stopping=True, random_state=42),
    }

# --- 4. Helper: Metric Calculation ---
def calc_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # Directional Accuracy
    true_dir = np.sign(y_true)
    pred_dir = np.sign(y_pred)
    acc = np.mean(true_dir == pred_dir) * 100
    return rmse, acc

# --- 5. MASSIVE TRAINING LOOP ---
print("Loading FULL model_features dataset...")
# Load the weekly tabular data created in 03A
df = pd.read_sql("SELECT * FROM model_features", con=db_engine)
df = df.sort_values(['ticker', 'date'])

# TARGET: ALL TICKERS
all_tickers = df['ticker'].unique().tolist()
print(f"Total Tickers to Process: {len(all_tickers)}")

# Resume Logic
results_file = output_dir / "05_Full_Scale_Traditional_Results.csv"
if results_file.exists():
    existing_df = pd.read_csv(results_file)
    completed_tickers = existing_df['Ticker'].unique().tolist()
    tickers_to_run = [t for t in all_tickers if t not in completed_tickers]
    print(f"Resuming... {len(completed_tickers)} done, {len(tickers_to_run)} remaining.")
else:
    tickers_to_run = all_tickers
    print("Starting fresh run.")

# Configuration
BATCH_SIZE = 50 
batch_results = []

print("Starting comparison: 8 Models x 3 Sentiment Variants x 4000+ Tickers")

for i, ticker in enumerate(tqdm(tickers_to_run, desc="Processing Tickers")):
    ticker_df = df[df['ticker'] == ticker].copy()
    
    # Minimum data check (skip ghosts)
    if len(ticker_df) < 50: continue
    
    # Train/Test Split (80/20)
    split_idx = int(len(ticker_df) * 0.8)
    train_df = ticker_df.iloc[:split_idx]
    test_df = ticker_df.iloc[split_idx:]
    
    y_train = train_df['target_return'].values
    y_test = test_df['target_return'].values
    
    # Iterate Variants (VADER, TextBlob, FinBERT)
    for source_name, cols in feature_variants.items():
        try:
            # Prepare X
            scaler = StandardScaler()
            X_train = scaler.fit_transform(train_df[cols])
            X_test = scaler.transform(test_df[cols])
            
            # --- A. Standard Scikit-Learn Models ---
            models = get_models()
            for model_name, model in models.items():
                try:
                    model.fit(X_train, y_train)
                    preds = model.predict(X_test)
                    
                    rmse, acc = calc_metrics(y_test, preds)
                    
                    batch_results.append({
                        "Ticker": ticker, "Model": model_name, "Source": source_name,
                        "RMSE": rmse, "Accuracy": acc, "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
                    })
                except: continue

            # --- B. ARIMAX (Statsmodels) ---
            try:
                # Identify exogenous columns (the sentiment cols)
                exog_cols = [c for c in cols if c not in base_features]
                
                if exog_cols:
                    exog_train = train_df[exog_cols]
                    exog_test = test_df[exog_cols]
                    
                    # Fit ARIMAX (5,1,0)
                    arima_model = ARIMA(train_df['target_return'], order=(5, 1, 0), exog=exog_train)
                    arima_fit = arima_model.fit()
                    
                    # Forecast
                    forecast = arima_fit.forecast(steps=len(y_test), exog=exog_test)
                    
                    rmse, acc = calc_metrics(y_test, forecast.values)
                    
                    batch_results.append({
                        "Ticker": ticker, "Model": "ARIMAX", "Source": source_name,
                        "RMSE": rmse, "Accuracy": acc, "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
                    })
            except: continue

        except: continue

    # --- SAVE BATCH ---
    if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(tickers_to_run):
        new_df = pd.DataFrame(batch_results)
        
        # CSV Save
        new_df.to_csv(results_file, mode='a', header=not results_file.exists(), index=False)
        
        # DB Save
        try:
            new_df.columns = [c.lower() for c in new_df.columns]
            new_df.to_sql('results_trad_source_ablation', con=db_engine, if_exists='append', index=False)
        except: pass
        
        print(f"Saved batch of {len(batch_results)} results.")
        batch_results = []

print("FULL SCALE TRADITIONAL ABLATION COMPLETE.")